<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/LTX_Director2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Cell 1: ComfyUI + LTX Director 2 Hotfix 必需节点
# ==========================================
import os, sys, subprocess
from pathlib import Path

print("=== 🚀 安装 ComfyUI + LTX Director 2 Hotfix 必需组件 ===")

BASE_DIR = Path("/content")
COMFY_DIR = BASE_DIR / "ComfyUI"
CUSTOM_NODES_DIR = COMFY_DIR / "custom_nodes"

def run(cmd, cwd=None, check=False):
    print(" ".join(map(str, cmd)))
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr.strip())
        if check:
            raise RuntimeError("命令失败: " + " ".join(map(str, cmd)))
    return r

os.chdir(BASE_DIR)

# ComfyUI
if not COMFY_DIR.exists():
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", str(COMFY_DIR)], check=True)
else:
    run(["git", "pull"], cwd=COMFY_DIR)

os.chdir(COMFY_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_transfer", "opencv-python-headless"], check=True)

CUSTOM_NODES_DIR.mkdir(parents=True, exist_ok=True)

def install_node(repo_url):
    name = repo_url.rstrip("/").split("/")[-1].replace(".git", "")
    path = CUSTOM_NODES_DIR / name

    if not path.exists():
        run(["git", "clone", repo_url, str(path)], check=True)
    else:
        run(["git", "pull"], cwd=path)

    req = path / "requirements.txt"
    if req.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

# ==========================================
# ✅ 本 workflow 必需节点
# 来自 JSON：LTXDirector / LTXDirectorGuide / LTXDirectorCropGuides
# VAELoaderKJ / ModelPreviewOverrideKJ
# LTXVConcatAVLatent / LTXVSeparateAVLatent / LTXVAudioVAEDecode / LTXVLatentUpsampler
# ==========================================
required_nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/WhatDreamsCost/WhatDreamsCost-ComfyUI.git",
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/Lightricks/ComfyUI-LTXVideo.git",
    "https://github.com/rgthree/rgthree-comfy.git",  # 可选 UI 增强
]

for repo in required_nodes:
    install_node(repo)

# ==========================================
# ❌ 本 workflow 不需要，保留注释
# ==========================================
unused_nodes = [
    # "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    # "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    # "https://github.com/cubiq/ComfyUI_essentials.git",
    # "https://github.com/yuvraj108c/ComfyUI-Video-Depth-Anything.git",
    # "https://github.com/evanspearman/ComfyMath.git",
    # "https://github.com/chrisgoringe/cg-use-everywhere.git",
    # "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    # "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    # "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git",
    # "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    # "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    # "https://github.com/DoctorDiffusion/ComfyUI-MediaMixer.git",
    # "https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git",
    # "https://github.com/11cafe/comfyui-workspace-manager.git",
    # "https://github.com/kijai/ComfyUI-PromptRelay.git",
    # "https://github.com/Saganaki22/ComfyUI-FishAudioS2.git",
    # "https://github.com/chflame163/ComfyUI_LayerStyle.git",
    # "https://github.com/yolain/ComfyUI-Easy-Use.git",
    # "https://github.com/BadCafeCode/masquerade-nodes-comfyui.git",
    # "https://github.com/ClownsharkBatwing/RES4LYF.git",
    # "https://github.com/GoogleCloudPlatform/comfyui-google-genmedia-custom-nodes.git",
]

# 兼容依赖
run([sys.executable, "-m", "pip", "install", "-q", "kornia==0.7.2"])
run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu", "scikit-image"])

# 可选加速，失败不影响
run([sys.executable, "-m", "pip", "install", "-q", "sageattention"])

print("\n✅ Cell 1 完成")

In [ ]:
# ==========================================
# Cell 2: 根据 LTX_Director_2_Workflow_Hotfix.json 下载必需模型
# ==========================================
import os, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download

print("=== 🚀 下载 LTX Director 2 Hotfix 必需模型 ===")

COMFY_DIR = Path("/content/ComfyUI")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("✅ 已读取 HF_TOKEN")
except Exception:
    pass

downloads = [
    # UNETLoader
    {
        "repo_id": "Kijai/LTX2.3_comfy",
        "filename": "diffusion_models/ltx-2.3-22b-distilled-1.1_transformer_only_fp8_scaled.safetensors",
        "final_dir": COMFY_DIR / "models" / "unet",
        "flatten": True,
    },

    # DualCLIPLoader
    {
        "repo_id": "Comfy-Org/ltx-2",
        "filename": "split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors",
        "final_dir": COMFY_DIR / "models" / "text_encoders",
        "flatten": True,
    },
    {
        "repo_id": "Kijai/LTX2.3_comfy",
        "filename": "text_encoders/ltx-2.3_text_projection_bf16.safetensors",
        "final_dir": COMFY_DIR / "models" / "text_encoders",
        "flatten": True,
    },

    # VAEs
    {
        "repo_id": "Kijai/LTX2.3_comfy",
        "filename": "vae/taeltx2_3.safetensors",
        "final_dir": COMFY_DIR / "models" / "vae",
        "flatten": True,
    },
    {
        "repo_id": "Kijai/LTX2.3_comfy",
        "filename": "vae/LTX23_video_vae_bf16.safetensors",
        "final_dir": COMFY_DIR / "models" / "vae",
        "flatten": True,
    },
    {
        "repo_id": "Kijai/LTX2.3_comfy",
        "filename": "vae/LTX23_audio_vae_bf16.safetensors",
        "final_dir": COMFY_DIR / "models" / "vae",
        "flatten": True,
    },

    # LatentUpscaleModelLoader
    {
        "repo_id": "Lightricks/LTX-2.3",
        "filename": "ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
        "final_dir": COMFY_DIR / "models" / "latent_upscale_models",
    },
]

# ❌ 本 workflow 不需要的模型，保留注释
unused_models = [
    # "ltx-2.3-22b-dev-fp8.safetensors",
    # "ltx-2.3-22b-dev.safetensors",
    # "ltx-2.3-22b-distilled-1.1.safetensors",
    # "ltx-2.3-22b-distilled-lora-384-1.1.safetensors",
    # "gemma_3_12B_it_fp8_scaled.safetensors",
    # "gemma_3_12B_it.safetensors",
    # "ltx-2.3-id-lora-talkvid-3k.safetensors",
    # "ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors",
    # "ltx-2.3-22b-ic-lora-hdr-0.9.safetensors",
    # "ltx-2.3-22b-ic-lora-motion-track-control-ref0.5.safetensors",
    # "ltx-2.3-22b-ic-lora-lipdub-0.9.safetensors",
    # "ip-adapter-faceid-plusv2_sdxl.bin",
    # "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors",
    # "s2-pro-fp8",
]

def download_model(task):
    final_dir = Path(task["final_dir"])
    final_dir.mkdir(parents=True, exist_ok=True)

    target_name = task.get("rename_to", Path(task["filename"]).name)
    final_path = final_dir / target_name

    if final_path.exists() and final_path.stat().st_size > 0:
        return f"⏩ 已存在: {target_name}"

    try:
        downloaded = hf_hub_download(
            repo_id=task["repo_id"],
            filename=task["filename"],
            repo_type="model",
            local_dir=str(final_dir),
        )
        downloaded = Path(downloaded)

        if task.get("flatten") or task.get("rename_to"):
            if downloaded.resolve() != final_path.resolve():
                if final_path.exists():
                    final_path.unlink()
                shutil.move(str(downloaded), str(final_path))

        return f"✅ 完成: {target_name}"

    except Exception as e:
        return f"❌ 失败: {target_name} | {e}"

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(download_model, x) for x in downloads]
    for f in as_completed(futures):
        print(f.result())

# 兼容：UNET 和 diffusion_models 都建软链接
unet_dir = COMFY_DIR / "models" / "unet"
diff_dir = COMFY_DIR / "models" / "diffusion_models"
diff_dir.mkdir(parents=True, exist_ok=True)

for f in unet_dir.glob("*.safetensors"):
    link = diff_dir / f.name
    if not link.exists():
        try:
            link.symlink_to(f)
            print(f"🔗 diffusion_models 软链接: {f.name}")
        except Exception:
            pass

print("\n🎉 Cell 2 完成")

In [ ]:
# ==========================================
# Cell 3: FRP 内网穿透，可选
# ==========================================
import os, subprocess
from pathlib import Path

print("=== 🌐 配置 FRP，可选 ===")

try:
    from google.colab import userdata
    VPS_IP = userdata.get("VPS_IP")
    FRP_TOKEN = userdata.get("FRP_TOKEN")
except Exception:
    VPS_IP = None
    FRP_TOKEN = None

if not VPS_IP or not FRP_TOKEN:
    print("⚠️ 没有 VPS_IP / FRP_TOKEN，跳过 FRP")
else:
    FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

    if not FRP_DIR.exists():
        subprocess.run(
            "wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content",
            shell=True,
            check=True
        )

    conf = f'''
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8090
'''
    (FRP_DIR / "frpc.toml").write_text(conf.strip(), encoding="utf-8")
    print("✅ FRP 配置完成")

In [ ]:
# ==========================================
# Cell 4: 启动 ComfyUI
# ==========================================
import os, subprocess, threading, time, configparser
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] Colab 保活中...")

threading.Thread(target=keep_alive, daemon=True).start()

# 启动 FRP
frpc = FRP_DIR / "frpc"
frpc_conf = FRP_DIR / "frpc.toml"

if frpc.exists() and frpc_conf.exists():
    def start_frpc():
        subprocess.run([str(frpc), "-c", str(frpc_conf)])

    threading.Thread(target=start_frpc, daemon=True).start()
    print("✅ FRP 已启动")
    print("👉 访问: http://cjp.usdream.dpdns.org:8090")
else:
    print("⚠️ 未启用 FRP")

# Manager private 模式，减少启动 fetch
manager_paths = [
    COMFY_DIR / "user" / "__manager" / "config.ini",
    COMFY_DIR / "user" / "default" / "ComfyUI-Manager" / "config.ini",
    COMFY_DIR / "custom_nodes" / "ComfyUI-Manager" / "config.ini",
]

for p in manager_paths:
    p.parent.mkdir(parents=True, exist_ok=True)
    cfg = configparser.ConfigParser()
    if p.exists():
        cfg.read(p)

    if "default" not in cfg:
        cfg["default"] = {}

    cfg["default"]["network_mode"] = "private"

    with open(p, "w") as f:
        cfg.write(f)

print("✅ Manager private 模式完成")

os.chdir(COMFY_DIR)
print("🚀 启动 ComfyUI...")
subprocess.run(["python", "main.py", "--dont-print-server"])